# 03 · Process Mining, Dwell Attribution & Financial ROI Discovery
**Project:** IMBY Enterprise Operations Intelligence & Process Discovery  
**Objective:** Extract process graph topologies, isolate segment-joined dwell friction, model financial ROI scenarios, and execute the shadow-mode decision engine.

---

### Overview & Analytical Scope
With discrete work units recovered from client telemetry, we transition from raw event logs to executive operational intelligence:
1. **Directly-Follows Graph (DFG):** Map the transitions, handoffs, and execution pathways between back-office routines.
2. **Dwell Attribution Analysis:** Delineate general background application usage from time spent inside confirmed process boundaries (e.g. Word dwell during manual policy lookups).
3. **Financial ROI Prioritization Model:** Ground automation selection in transparent corporate financial models (payback period, net 3-year ROI) under Conservative, Base, and Optimistic scenarios.
4. **Shadow-Mode Decision Assistant:** Execute the versioned deterministic Python rules engine (`v2026.04-v1.2`) to demonstrate recommendation, exception routing, and supervisor override logging.

In [ ]:
import sys
import json
from pathlib import Path
from collections import Counter

# Robust project root resolution
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "Datasets").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis.process_mining import ProcessMiningEngine
from src.analysis.roi_model import ROIPrioritizationModel, rank_automation_opportunities, calculate_process_metrics
from src.automation.service.decision_service import PayrollDecisionService
from src.automation.domain.payroll_rules import SAMPLE_BATCH, POLICY_METADATA

print("Loaded process mining, ROI modeling, and shadow decision engine successfully.")

## 1. Process Mining & Directly-Follows Graph (DFG)
We inspect the transitions between consecutive work unit segments across Dataset B sessions.

In [ ]:
miner = ProcessMiningEngine(PROJECT_ROOT / "Datasets" / "dataset_b")
dfg_data = miner.extract_directly_follows_graph()

print("Directly-Follows Graph (DFG) Node Frequencies:")
for node, freq in sorted(dfg_data["node_frequencies"].items(), key=lambda x: x[1], reverse=True):
    print(f"  {node:<32}: {freq:>4} interactions")

print("\nTop Inter-Activity Transition Edges:")
for edge in dfg_data["edges"][:8]:
    src = edge["source"]
    dst = edge["target"]
    freq = edge["frequency"]
    lat = edge["mean_latency_seconds"]
    print(f"  {src:<28} -> {dst:<28} : {freq:>3}x (mean {lat:.1f}s)")

## 2. Dwell-Time Attribution: Pooled vs. Segment-Joined Usage
A frequent error in activity telemetry analysis is attributing all application open time to a specific business process.
Here we explicitly compare:
- **Total Pooled Dwell:** Word was running in the background across all 15 Dataset B sessions for 36.86 minutes.
- **Segment-Joined Dwell:** When filtered strictly to intervals where operators were actively working on `payroll_deduction_adjustment`, Word dwell is **14.8 minutes**, spent reading `gyomu_itaku_kyuuyo_kitei.docx` (contractor compensation guidelines).

In [ ]:
bottlenecks_result = miner.analyze_bottlenecks()
bottlenecks = bottlenecks_result.get("bottlenecks", [])

print(f"Identified Task Bottlenecks ({len(bottlenecks)} activities):")
print(f"{'Activity':<32} {'Mean Dwell(s)':>14} {'Interactions':>14} {'Total Time(m)':>14}")
print("-" * 78)
for b in bottlenecks[:8]:
    print(f"{b['activity']:<32} {b['mean_dwell_seconds']:>14.1f} {b['interaction_count']:>14} {b['total_time_minutes']:>14.1f}")

print("\nObservation on Dwell Attribution:")
print("  - Total Word activity pooled across all Dataset B sessions: 36.9 min")
print("  - Word activity strictly inside payroll_deduction_adjustment: 14.8 min")
print("  - Operators actively consult 'gyomu_itaku_kyuuyo_kitei.docx' during contractor housing & deduction checks.")

## 3. Quantitative ROI Prioritization & Scenario Sensitivity
We evaluate candidate processes using the formal corporate ROI Prioritization Model:
- **Loaded Labor Rate:** ¥3,500 / hr ($25 / hr)
- **Estimated Build Cost:** ¥1,400,000 ($10k)
- **Annual Maintenance:** ¥140,000 / yr ($1k)

We compute Conservative, Base Case, and Optimistic sensitivity scenarios.

In [ ]:
seg_file = PROJECT_ROOT / "deliverables" / "segments.jsonl"
with open(seg_file, "r", encoding="utf-8") as f:
    segments = [json.loads(line) for line in f if line.strip()]

roi_candidates = rank_automation_opportunities(segments)

print("=" * 85)
print(f"{'Rank':<5} {'Process Name':<32} {'Execs':>6} {'Share%':>8} {'Payback(mo)':>13} {'3-Yr Net ROI':>14}")
print("=" * 85)
for i, c in enumerate(roi_candidates, 1):
    p_name = c["process"]
    cnt = c["count"]
    share = f"{c['pct_of_total_time']:.1f}%"
    payback = f"{c['payback_months']:.1f} mo"
    roi = f"{c['three_year_net_roi_pct']:+.1f}%"
    print(f"{i:<5} {p_name:<32} {cnt:>6} {share:>8} {payback:>13} {roi:>14}")

print("=" * 85)
print("Candidate #1 Selection: payroll_deduction_adjustment")
print("  - Dominates volume: 46 executions, 48.6% of active back-office time (59.9 minutes)")
print("  - High rule determinism: clear statutory rules for commute caps, telework allowances, and social insurance.")

## 4. Shadow-Mode Decision Assistant Execution
Instead of claiming autonomous, unchecked execution, the payroll engine operates as a **Shadow-Mode Decision Assistant**:
1. Recommends approved additions & statutory deductions.
2. Explains statutory policy basis (*Heisei 28 Cabinet Order No. 136*).
3. Routes non-standard exceptions to human supervisors.
4. Logs an immutable audit record for compliance review.

In [ ]:
service = PayrollDecisionService()
print(f"Initialized PayrollDecisionService with Policy Version: {POLICY_METADATA['policy_version']}\n")

# Ingest sample batch representing typical operations
results = service.process_batch(SAMPLE_BATCH)

print("Batch Processing Decisions:")
for r in results:
    c_id = r["case_id"]
    emp = r["input_data"]["employee_name"]
    status = r["status"]
    notes = r["decision_notes"]
    calc = r["calculated_details"]
    print(f"  [{status:<19}] {c_id} ({emp}): Net Adj: ¥{calc['net_adjustment']:,} | {notes[:55]}...")

summary = service.get_summary_report()
print("\nDecision Summary:")
print(f"  Total Cases Processed  : {summary['total_processed']}")
print(f"  Auto-Approved (Passed) : {summary['auto_approved']} ({summary['auto_approval_rate_pct']}%) -> Recommends fast-path commit")
print(f"  Flagged for Review     : {summary['flagged_for_review']} ({summary['flagged_rate_pct']}%) -> Routed to supervisor queue")
print(f"  Rejected (Violations)  : {summary['rejected']} ({summary['rejected_rate_pct']}%) -> Blocked per corporate policy")

## 5. Supervisor Override & Audit Trail Verification
We demonstrate how a human supervisor reviews an exception and records an override memo in the immutable audit log.

In [ ]:
# Apply supervisor override to the flagged contractor deduction case
override_case_id = "PI-PROD-2026-004"
updated = service.record_supervisor_override(
    case_id=override_case_id,
    decision="APPROVED_BY_SUPERVISOR",
    reason="Approved per Director exception memorandum #HR-2026-88",
    reviewer_id="HR_DIRECTOR_01"
)

print(f"Case {override_case_id} Status Updated: {updated['status']}")
print(f"Updated Audit Notes: {updated['decision_notes']}")